# MPHY Final Project Analysis

This notebook is a lightweight analysis entry point for reviewing the saved workflow outputs after running the evaluation pipeline.

## What to inspect

- `outputs/workflows/<timestamp>/<stage>/summary.json`
- `outputs/workflows/<timestamp>/<stage>/per_class_metrics_<method>.json`
- `outputs/figures/evaluation_summary.md`

The code below looks for the most recent workflow summary and prints the headline metrics for each stage.

In [1]:
from pathlib import Path
import json

repo_root = Path.cwd()
if not (repo_root / "outputs").exists():
    repo_root = repo_root.parent

workflows_root = repo_root / "outputs" / "workflows"
workflow_dirs = sorted([p for p in workflows_root.iterdir() if p.is_dir()])
latest = workflow_dirs[-1]
latest

PosixPath('/Users/zizhuoliang/Desktop/MPHY-FINAL/outputs/workflows/20260506T065053Z')

In [2]:
for stage in ["development", "final_gold", "final_asr"]:
    summary_path = latest / stage / "summary.json"
    if not summary_path.exists():
        continue
    summary = json.loads(summary_path.read_text())
    print(stage)
    print("  transcript source:", summary["transcript_source"])
    for method, metrics in summary["method_metrics"].items():
        print(
            f"  {method}: acc={metrics['accuracy']:.3f}, "
            f"macro_f1={metrics['macro_f1']:.3f}, "
            f"review_rate={metrics['human_review_rate']:.3f}"
        )
    print()


development
  transcript source: gold
  few_shot: acc=0.960, macro_f1=0.959, review_rate=0.067
  zero_shot: acc=0.933, macro_f1=0.927, review_rate=0.080

final_gold
  transcript source: gold
  few_shot: acc=0.924, macro_f1=0.923, review_rate=0.042
  zero_shot: acc=0.896, macro_f1=0.892, review_rate=0.064

final_asr
  transcript source: asr
  few_shot: acc=0.890, macro_f1=0.890, review_rate=0.062
  zero_shot: acc=0.836, macro_f1=0.834, review_rate=0.104



In [3]:
summary_md = repo_root / "outputs" / "figures" / "evaluation_summary.md"
print(summary_md.read_text())

# Model Performance Evaluation

## Final Gold (held-out test transcripts, gold text input)
- Sample: **n = 500** stratified across 25 complaint labels (~20 / class)
- **Zero-shot**: accuracy 89.6% (95% CI 87.0%–91.8%), macro-F1 0.892 (95% CI 0.863–0.913), top-3 acc 96.8%
- **Few-shot**: accuracy 92.4% (95% CI 90.2%–94.6%), macro-F1 0.923 (95% CI 0.899–0.944), top-3 acc 98.2%
- **Few-shot lift over zero-shot**: +2.8 pp accuracy, +0.031 macro-F1
- **Cost**: ~$1.612 (0.771 Zero-shot + 0.840 Few-shot)

## Final ASR (end-to-end audio → Whisper STT → classifier)
- Sample: **n = 500** stratified across the 25 labels
- **STT fidelity** (Whisper large-v3-turbo): exact-match 61.8%, avg similarity 0.970, avg word overlap 0.886
- **Zero-shot** (over ASR): accuracy 83.6% (95% CI 79.8%–86.6%), macro-F1 0.834
- **Few-shot** (over ASR): accuracy 89.0% (95% CI 86.2%–91.6%), macro-F1 0.890
- **Gold vs ASR stage-level gap (few-shot)**: 92.4% vs 89.0% (difference 3.4 pp; separate stratified samples, overl